# Required Assignment 31 – Natural Language Processing
## Building and Evaluating NLP Applications Using Hugging Face Transformers
**Name:** Yundi Li
This notebook completes all TODO sections: Hugging Face pipelines, tokenisers, Transformer models, sequence classification and architectures.

In [1]:
# !pip install "transformers<5" torch
import os
os.environ["TQDM_DISABLE"] = "1"
import warnings, torch, transformers
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
from transformers import pipeline, set_seed
print("transformers", transformers.__version__, "| torch", torch.__version__)

transformers 4.57.6 | torch 2.14.0


## Part A – NLP Pipelines
### A1. Sentiment analysis

In [2]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
texts = ["The lecture was engaging and easy to follow.", "The notebook is confusing."]
results = classifier(texts)
for t, r in zip(texts, results):
    print(f"{t!r:50} -> {r['label']} ({r['score']:.4f})")

'The lecture was engaging and easy to follow.'     -> POSITIVE (0.9997)
'The notebook is confusing.'                       -> NEGATIVE (0.9994)


**Task:** assigns a polarity label (POSITIVE/NEGATIVE) with a confidence score to each text. **Application:** text classification – e.g. customer-review/feedback monitoring, social media opinion mining.

### A2. Fill-mask

In [3]:
unmasker = pipeline("fill-mask", model="bert-base-uncased")
for r in unmasker("This course will teach you all about [MASK] models.", top_k=5):
    print(f"{r['token_str']:>12}  {r['score']:.4f}  | {r['sequence']}")

        role  0.3366  | this course will teach you all about role models.
    business  0.0837  | this course will teach you all about business models.
         the  0.0572  | this course will teach you all about the models.
        your  0.0439  | this course will teach you all about your models.
     fashion  0.0267  | this course will teach you all about fashion models.


**Task:** predicts the most likely tokens for the `[MASK]` position using context on both sides (the masked-language-modelling objective BERT is pretrained on). **Application:** language modelling / autocomplete, query suggestion, data augmentation and probing what a model has learned.

### A3. Named entity recognition (NER)

In [4]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="first")
for e in ner("My name is Sylvain and I work at Hugging Face in Brooklyn."):
    print(f"{e['word']:15} {e['entity_group']:5} {e['score']:.4f}")

Sylvain         PER   0.9986
Hugging Face    ORG   0.9777
Brooklyn        LOC   0.9971


**Task:** token classification – finds spans that are persons (PER), organisations (ORG), locations (LOC) and misc entities; `aggregation_strategy='first'` merges sub-word pieces (e.g. S + ##yl + ##va + ##in) into whole entities. **Application:** information extraction – e.g. extracting names, companies and places from documents, CVs, news or support tickets.

### A4. Question answering

In [5]:
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
r = qa(question="Where do I work?", context="My name is Sylvain and I work at Hugging Face in Brooklyn.")
print(r)

{'score': 0.6385905742645264, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}


**Task:** extractive QA – predicts the start and end positions of the answer span inside the supplied context (it does not generate new text). **Application:** reading comprehension, document search, FAQ/help-desk bots.

In [ ]:
for q in ["Where is the office located?", "What is my name?"]:
    r = qa(question=q, context="My name is Sylvain and I work at Hugging Face in Brooklyn.")
    print(f"{q:30} -> {r['answer']!r} (score {r['score']:.4f}, chars {r['start']}-{r['end']})")

Different questions retrieve different spans from the same context, confirming the model locates the answer by position rather than generating it.

### A5. Text generation

In [6]:
set_seed(42)
generator = pipeline("text-generation", model="gpt2")
outs = generator("In this course, we will teach you how to", max_new_tokens=30, num_return_sequences=2, do_sample=True, pad_token_id=50256)
for i, o in enumerate(outs, 1):
    print(f"[{i}] {o['generated_text']}\n")

[1] In this course, we will teach you how to use the Google Assistant to create a list of your favorite hashtags and share them with the world. This course is designed for students to gain an understanding

[2] In this course, we will teach you how to build a simple web application that is easy to use for web development. We will provide you with some of the features that you will need in order to



**Task:** auto-regressive generation – GPT-2 repeatedly predicts the next token given all previous tokens. **Application:** natural language generation – drafting content, chatbots, story/code completion. Note the output is fluent but not necessarily factual or on-topic, illustrating hallucination risk.

### A6. Probing bias with fill-mask
Swapping only the occupation word shows whether the model associates professions with a gender – evidence for the bias discussion in Part D.

In [ ]:
for occ in ["nurse", "engineer", "doctor", "receptionist"]:
    preds = unmasker(f"The {occ} said that [MASK] would be late.", targets=["he", "she"])
    scores = {p["token_str"]: p["score"] for p in preds}
    print(f"{occ:13} he={scores['he']:.4f}  she={scores['she']:.4f}  ratio he/she={scores['he']/scores['she']:.2f}")

**Observation:** *(update after running)* compare the he/she probabilities across occupations. A shift in the ratio driven only by the occupation word reflects gender stereotypes absorbed from the pretraining corpus, not anything in the sentence itself.

### Part A summary
| Pipeline | Task performed | NLP application | Model type |
|---|---|---|---|
| sentiment-analysis | Sequence classification (pos/neg) | Opinion mining, review analysis | Encoder (DistilBERT) |
| fill-mask | Predict masked token | Language modelling / autocomplete | Encoder (BERT) |
| ner | Token classification | Information extraction | Encoder (BERT) |
| question-answering | Extract answer span from context | Reading comprehension, search | Encoder (DistilBERT) |
| text-generation | Next-token generation | Natural language generation | Decoder (GPT-2) |

## Part B – Tokenizer, AutoModel and Sequence Classification
### B1. Tokenisation

In [7]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sentences = ["This course is excellent.", "I hate waiting in long queues at the bank!"]
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
print("input_ids:\n", inputs["input_ids"])
print("attention_mask:\n", inputs["attention_mask"])
for i, ids in enumerate(inputs["input_ids"]):
    print(f"tokens[{i}]:", tokenizer.convert_ids_to_tokens(ids))

input_ids:
 tensor([[  101,  2023,  2607,  2003,  6581,  1012,   102,     0,     0,     0,
             0,     0,     0],
        [  101,  1045,  5223,  3403,  1999,  2146, 24240,  2015,  2012,  1996,
          2924,   999,   102]])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tokens[0]: ['[CLS]', 'this', 'course', 'is', 'excellent', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
tokens[1]: ['[CLS]', 'i', 'hate', 'waiting', 'in', 'long', 'queue', '##s', 'at', 'the', 'bank', '!', '[SEP]']


### B2. Decoding back to text

In [8]:
for ids in inputs["input_ids"]:
    print("with special tokens   :", tokenizer.decode(ids))
    print("without special tokens:", tokenizer.decode(ids, skip_special_tokens=True))
print("[CLS] id =", tokenizer.cls_token_id, "| [SEP] id =", tokenizer.sep_token_id, "| [PAD] id =", tokenizer.pad_token_id)

with special tokens   : [CLS] this course is excellent. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
without special tokens: this course is excellent.
with special tokens   : [CLS] i hate waiting in long queues at the bank! [SEP]
without special tokens: i hate waiting in long queues at the bank!
[CLS] id = 101 | [SEP] id = 102 | [PAD] id = 0


**Observations**
- The tokenizer lower-cases text (uncased model), splits punctuation, and adds `[CLS]` (id 101) at the start and `[SEP]` (id 102) at the end.
- The shorter sentence is padded with `[PAD]` (id 0); its `attention_mask` is 0 at those positions so the model ignores them.
- Decoding reverses the mapping; `skip_special_tokens=True` recovers the original (lower-cased) text.

### B3. AutoModel – hidden states (no task head)

In [9]:
base_model = AutoModel.from_pretrained(checkpoint)
with torch.no_grad():
    hidden = base_model(**inputs).last_hidden_state
print("last_hidden_state shape (batch, seq_len, hidden_size):", tuple(hidden.shape))

last_hidden_state shape (batch, seq_len, hidden_size): (2, 13, 768)


`AutoModel` returns contextual embeddings: one 768-dimensional vector per token. It has no classification head, so these vectors must be fed into a head to perform a task.

### B4. AutoModelForSequenceClassification – logits, softmax, labels

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
with torch.no_grad():
    outputs = model(**inputs)
print("logits:\n", outputs.logits)
probabilities = torch.softmax(outputs.logits, dim=-1)
print("probabilities:\n", probabilities)
print("label map:", model.config.id2label)
for s, p in zip(sentences, probabilities):
    idx = int(p.argmax())
    print(f"{s!r:48} -> {model.config.id2label[idx]} ({p[idx]:.4f})")

logits:
 tensor([[-4.2635,  4.6273],
        [ 3.5489, -2.9267]])
probabilities:
 tensor([[1.3764e-04, 9.9986e-01],
        [9.9846e-01, 1.5382e-03]])
label map: {0: 'NEGATIVE', 1: 'POSITIVE'}
'This course is excellent.'                      -> POSITIVE (0.9999)
'I hate waiting in long queues at the bank!'     -> NEGATIVE (0.9985)


## Reflection
1. **Why is tokenisation necessary?** Neural networks only process numbers. Tokenisation splits raw text into units from a fixed vocabulary (sub-words via WordPiece here) and maps each to an integer ID that indexes an embedding table. Sub-word tokenisation keeps the vocabulary small while still handling rare/unseen words (e.g. `queues` → known pieces), and it must match exactly the tokenizer the model was pretrained with.
2. **Purpose of `[CLS]` and `[SEP]`.** `[CLS]` is prepended to every input; through self-attention its final hidden state aggregates information from the whole sequence and is used by the classification head as the sentence representation. `[SEP]` marks the end of a sequence and separates two segments in sentence-pair tasks (e.g. question + context in QA, premise + hypothesis in NLI).
3. **What are logits?** The raw, unnormalised scores output by the final linear layer, one per class. They can be any real number and do not sum to 1; the larger logit indicates the preferred class.
4. **Why is Softmax applied?** Softmax exponentiates and normalises the logits so they lie in [0,1] and sum to 1, giving interpretable class probabilities/confidence. Argmax of the probabilities (identical to argmax of logits) gives the predicted label, mapped with `model.config.id2label`.

## Part C – Transformer Architectures
- **Encoder-only (BERT, DistilBERT, RoBERTa):** bidirectional self-attention, every token sees left and right context → understanding tasks: sentiment/text classification, NER, extractive QA, fill-mask.
- **Decoder-only (GPT family):** masked (causal) self-attention, each token sees only previous tokens → generative tasks: text generation, chat, completion.
- **Encoder-decoder (T5, BART, MarianMT):** encoder reads input; decoder generates output attending to it via cross-attention → sequence-to-sequence: translation, summarisation, generative QA.
- **Attention:** each token builds a query, key and value; softmax(QKᵀ/√d)·V weights every other token by relevance, giving context-dependent representations (e.g. "bank" near "river" vs "money"). Multi-head attention learns several relation types in parallel and enables parallel training over long ranges.

## Part D – Critical Evaluation
See the submitted Word document for the full ~500-word discussion.